# Example pipeline data preparation.

## Prepare notebook.

### Import the libraries.

In [ ]:
from sc_flow.data import DataManager
from sc_flow.data.sim import get_dummy_adata
from sc_flow.data.samplers import FTrainSampler, FValidationSampler

### Generate dummy data.

In [ ]:
BIG = False
if BIG:
    n_obs_pert = 1_000_000
    n_obs_ctrl = 500_000
else:
    n_obs_pert = 10000
    n_obs_ctrl = 5000
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl)
adata

## Case 1. Loading only state data.

### Initialize data manager and get data.

In [ ]:
dm = DataManager(
    sample_rep="X_repr",
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(
    train_collection,  # the tree containing the data
    lambda x: x,  # the function to process the nodes
    n_groups=1,  # the number of nodes to load
    replace_groups=True,  # whether to sample nodes with replacement
    replace_samples=True,  # whether to sample observations from nodes with replacement
    use_groups_weights=True,  # whether to weight sampling by frequency
)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

## Case 2: Grouping data based on source split.

### Initialize data manager and get data.

In [ ]:
dm = DataManager(
    sample_rep="X_repr",
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

## Case 2: Grouping data based on source split and condition.


### Initialize data manager and get data.

In [ ]:
dm = DataManager(
    sample_rep="X_repr",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

## Case 3: Grouping data based on source split and condition with controls.


### Initialize data manager and get data.

In [ ]:
dm = DataManager(
    sample_rep="X_tgt",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
    conditions_covariates=["paired_condition"],
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
    control_values_dict={"drug": "control", "ko": "control"},
    source_rep="X_src",
    n_shared_dims=10,
)
train_collection = dm.compile_adata(adata)

train_sampler = FTrainSampler(train_collection, lambda x: x, n_groups=1, replace_groups=True, replace_samples=True)
val_sampler = FValidationSampler(train_collection, lambda x: x, replace_groups=True, replace_samples=True)
batch = train_sampler.sample()
batch[0], val_sampler[0]

### Sampling a bunch of times.

In [ ]:
from tqdm import tqdm

for _ in tqdm(range(1000)):
    batch = train_sampler.sample()